Cell 1 — Imports for Hybrid Method 2


In [1]:
import os
import time
import numpy as np
import pandas as pd
import faiss

from typing import Dict, List, Tuple
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import normalize

Cell 2 — File paths for the 6 single-method result files


In [2]:
emb_folder = "\Embeddings"
EMBEDDING_FILES = {
    "bert_finetuned": emb_folder + "\\bert_finetuned_embeddings.xlsx",
    "gemini": emb_folder + "\\Gemini_Embedding.xlsx",
    "qwen3_8b": emb_folder + "\\Qwen3_Embedding_8B.xlsx",
    "sbert": emb_folder + "\\SBERT_Embedding_2_classification.xlsx",
}


SIM_METRICS = ["cosine", "dot", "neg_l2"]
TOPK_LIST = [1, 5, 10, 20]

# Weighted RRF without Flat (Flat is kept only as exact-search baseline)
RRF_K = 60
FUSION_DEPTH = 100

# soft filtering params
CLASS_TOPK = 3
BETA = 1.0
N_SPLITS = 5
RANDOM_STATE = 42

# IVF params
IVF_NLIST = 128
IVF_NPROBE = 10

# PQ params
PQ_NLIST = 128
PQ_NBITS = 8
PQ_NPROBE = 10

# HNSW params
HNSW_M = 32
HNSW_EF_CONSTRUCTION = 200
HNSW_EF_SEARCH = 64

# include oracle static filtered run?
INCLUDE_FILTERED_STATIC = True

# output files
OUT_SUMMARY_XLSX = "hybrid_weighted_rrf_no_flat_summary.xlsx"
OUT_RUN_CSV = "hybrid_weighted_rrf_no_flat_run.csv"

# for computing method weights from scratch
WEIGHT_TARGET_METRIC = "cosine"
WEIGHT_TARGET_K = 10

# Hybrid method identity
HYBRID_METHOD_NAME = "hybrid_weighted_rrf_no_flat"

# Flat is intentionally excluded from the proposed scalable hybrid fusion.
# It can still be evaluated separately as the exact-search baseline.
FUSION_METHODS = [
    "soft_filtering",
    "ivf",
    "ivfpq",
    "hnsw",
    "filtered_static"
]


Cell 3 — Loader


In [3]:
def is_numeric_col_name(c):
    if isinstance(c, (int, np.integer)):
        return True
    s = str(c)
    return s.isdigit()

def load_embedding_xlsx(path):
    df = pd.read_excel(path, engine="openpyxl")

    # label column
    label_candidates = [c for c in df.columns if str(c).lower() in ("label", "y", "class")]
    if not label_candidates:
        raise ValueError(f"{path} -> label column not found")
    label_col = label_candidates[0]

    # id column
    id_candidates = [c for c in df.columns if str(c).lower() in ("filename", "file", "text_file", "id", "file_id")]
    if id_candidates:
        preferred = [c for c in id_candidates if str(c).lower() in ("filename", "file", "text_file")]
        id_col = preferred[0] if preferred else id_candidates[0]
    else:
        non_num = [c for c in df.columns if c != label_col and not is_numeric_col_name(c)]
        if not non_num:
            raise ValueError(f"{path} -> id column not found")
        id_col = non_num[0]

    # embedding columns
    emb_cols = [c for c in df.columns if c not in (label_col, id_col) and (
        is_numeric_col_name(c) or str(c).lower().startswith("e") or str(c).lower().startswith("emb")
    )]

    if not emb_cols:
        raise ValueError(f"{path} -> embedding columns not found")

    X = df[emb_cols].to_numpy(dtype=np.float32)
    y = df[label_col].to_numpy()
    ids = df[id_col].astype(str).to_numpy()

    return df, X, y, ids

Cell 4 — Evaluation Metrics


In [4]:
def precision_at_k(rels, k):
    return float(np.sum(rels[:k])) / float(k)

def recall_at_k(rels, k, total_rel):
    if total_rel <= 0:
        return 0.0
    return float(np.sum(rels[:k])) / float(total_rel)

def dcg_at_k(rels, k):
    rels = rels[:k]
    denom = np.log2(np.arange(2, len(rels) + 2))
    return float(np.sum(rels / denom))

def ndcg_at_k(rels, k):
    dcg = dcg_at_k(rels, k)
    ideal = np.sort(rels)[::-1]
    idcg = dcg_at_k(ideal, k)
    return 0.0 if idcg == 0 else dcg / idcg

def mrr_at_k(rels, k):
    rels = rels[:k]
    idx = np.where(rels == 1)[0]
    if len(idx) == 0:
        return 0.0
    return 1.0 / float(idx[0] + 1)

Cell 5 — Similarity helpers


In [5]:
def prepare_vectors(X, metric):
    if metric == "cosine":
        return normalize(X, axis=1).astype(np.float32)
    return np.asarray(X, dtype=np.float32)

def scores_for_metric(q, X, metric):
    """
    Return scores where larger is better.
    cosine / dot : higher is better
    neg_l2       : negative squared L2 distance, so higher is better
    """
    if metric in ("cosine", "dot"):
        return X @ q

    if metric == "neg_l2":
        x2 = np.sum(X * X, axis=1)
        q2 = float(np.sum(q * q))
        return -(x2 + q2 - 2.0 * (X @ q))

    raise ValueError(f"Unknown metric: {metric}")

Cell 6 — Soft classification probabilities (for classifier-guided retrieval)


In [6]:
def cv_soft_probabilities(X, y, n_splits=5, random_state=42):
    classes = np.unique(y)
    class_to_index = {c: i for i, c in enumerate(classes)}

    N = len(y)
    C = len(classes)

    P = np.zeros((N, C), dtype=np.float32)

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    for train_idx, test_idx in skf.split(X, y):
        Xtr, Xte = X[train_idx], X[test_idx]
        ytr = y[train_idx]

        clf = LogisticRegression(max_iter=2000)
        clf.fit(Xtr, ytr)

        proba = clf.predict_proba(Xte)

        for j, c in enumerate(clf.classes_):
            P[test_idx, class_to_index[c]] = proba[:, j]

    return P, classes, {"class_to_index": class_to_index}

Cell 7 — Utility to convert neighbors into ranked-list rows


In [7]:
def build_run_rows_from_neighbors(method_name, metric, query_idx, nbrs, ids, y, depth):
    rows = []
    rank = 1
    for j in nbrs[:depth]:
        if j < 0:
            continue
        rows.append({
            "method": method_name,
            "metric": metric,
            "query_id": ids[query_idx],
            "query_label": int(y[query_idx]),
            "doc_id": ids[j],
            "doc_label": int(y[j]),
            "rank": rank,
            "relevance": int(y[j] == y[query_idx]),
        })
        rank += 1
    return rows

Cell 8 — FAISS Flat ranked-list builder


In [8]:
def build_run_flat(X, y, ids, metric, depth):
    X_use = prepare_vectors(X, metric)
    N, d = X_use.shape

    if metric == "neg_l2":
        index = faiss.IndexFlatL2(d)
    else:
        index = faiss.IndexFlatIP(d)

    index.add(X_use)

    search_k = min(N, depth + 1)
    D, I = index.search(X_use, search_k)

    rows = []
    for i in range(N):
        nbrs = I[i]
        nbrs = nbrs[(nbrs >= 0) & (nbrs != i)]  # remove self
        rows.extend(build_run_rows_from_neighbors("flat", metric, i, nbrs, ids, y, depth))

    return pd.DataFrame(rows)

Cell 9 — IVF ranked-list builder


In [9]:
def build_run_ivf(X, y, ids, metric, depth, nlist=128, nprobe=10):
    X_use = prepare_vectors(X, metric)
    N, d = X_use.shape

    if metric == "neg_l2":
        quantizer = faiss.IndexFlatL2(d)
        index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_L2)
    else:
        quantizer = faiss.IndexFlatIP(d)
        index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)

    if not index.is_trained:
        index.train(X_use)

    index.add(X_use)
    index.nprobe = nprobe

    search_k = min(N, depth + 1)
    D, I = index.search(X_use, search_k)

    rows = []
    for i in range(N):
        nbrs = I[i]
        nbrs = nbrs[(nbrs >= 0) & (nbrs != i)]
        rows.extend(build_run_rows_from_neighbors("ivf", metric, i, nbrs, ids, y, depth))

    return pd.DataFrame(rows)

Cell 10 — IVFPQ ranked-list builder


In [11]:
def choose_pq_m(d):
    # Prefer sub-dim 8, then 16, then 4
    for sd in [8, 16, 4]:
        if d % sd == 0:
            return d // sd
    for M in range(min(96, d), 1, -1):
        if d % M == 0:
            return M
    raise ValueError(f"Cannot find suitable PQ_M for d={d}")

def build_run_ivfpq(X, y, ids, metric, depth, nlist=128, nprobe=10, nbits=8):
    X_use = prepare_vectors(X, metric)
    N, d = X_use.shape
    M = choose_pq_m(d)

    if metric == "neg_l2":
        quantizer = faiss.IndexFlatL2(d)
        index = faiss.IndexIVFPQ(quantizer, d, nlist, M, nbits, faiss.METRIC_L2)
    else:
        quantizer = faiss.IndexFlatIP(d)
        index = faiss.IndexIVFPQ(quantizer, d, nlist, M, nbits, faiss.METRIC_INNER_PRODUCT)

    if not index.is_trained:
        index.train(X_use)

    index.add(X_use)
    index.nprobe = nprobe

    search_k = min(N, depth + 1)
    D, I = index.search(X_use, search_k)

    rows = []
    for i in range(N):
        nbrs = I[i]
        nbrs = nbrs[(nbrs >= 0) & (nbrs != i)]
        rows.extend(build_run_rows_from_neighbors("ivfpq", metric, i, nbrs, ids, y, depth))

    return pd.DataFrame(rows)

Cell 11 — HNSW ranked-list builder


In [12]:
def build_run_hnsw(X, y, ids, metric, depth, M=32, efc=200, efs=64):
    X_use = prepare_vectors(X, metric)
    N, d = X_use.shape

    if metric == "neg_l2":
        index = faiss.IndexHNSWFlat(d, M, faiss.METRIC_L2)
    else:
        index = faiss.IndexHNSWFlat(d, M, faiss.METRIC_INNER_PRODUCT)

    index.hnsw.efConstruction = int(efc)
    index.hnsw.efSearch = int(efs)

    index.add(X_use)

    search_k = min(N, depth + 1)
    D, I = index.search(X_use, search_k)

    rows = []
    for i in range(N):
        nbrs = I[i]
        nbrs = nbrs[(nbrs >= 0) & (nbrs != i)]
        rows.extend(build_run_rows_from_neighbors("hnsw", metric, i, nbrs, ids, y, depth))

    return pd.DataFrame(rows)

Cell 12 — Filtered Static ranked-list builder


In [13]:
def build_run_filtered_static(X, y, ids, metric, depth):
    X_use = prepare_vectors(X, metric)
    rows = []

    classes = np.unique(y)
    class_to_idx = {c: np.where(y == c)[0] for c in classes}

    for i in range(len(ids)):
        cand = class_to_idx[y[i]]
        cand = cand[cand != i]  # remove self

        if cand.size == 0:
            continue

        q = X_use[i]
        Xc = X_use[cand]

        scores = scores_for_metric(q, Xc, metric)

        take = min(depth, len(cand))
        top_local = np.argpartition(-scores, take - 1)[:take]
        top_local = top_local[np.argsort(-scores[top_local])]
        nbrs = cand[top_local]

        rows.extend(build_run_rows_from_neighbors("filtered_static", metric, i, nbrs, ids, y, depth))

    return pd.DataFrame(rows)

Cell 13 — Classifier-guided / Soft Filtering ranked-list builder


In [14]:
def build_run_soft_filtering(
    X, y, ids, metric, depth,
    class_topk=3, beta=1.0,
    n_splits=5, random_state=42
):
    P, classes, meta = cv_soft_probabilities(
        X, y,
        n_splits=n_splits,
        random_state=random_state
    )
    class_to_index = meta["class_to_index"]

    X_use = prepare_vectors(X, metric)
    rows = []

    for i in range(len(ids)):
        p = P[i]

        k_eff = min(class_topk, len(p))
        top_class_idx = np.argpartition(-p, k_eff - 1)[:k_eff]
        top_class_idx = top_class_idx[np.argsort(-p[top_class_idx])]
        top_classes = classes[top_class_idx]

        cand = np.where(np.isin(y, top_classes))[0]
        cand = cand[cand != i]  # remove self

        if cand.size == 0:
            continue

        q = X_use[i]
        Xc = X_use[cand]

        scores = scores_for_metric(q, Xc, metric)

        cand_labels = y[cand]
        class_probs = np.array(
            [p[class_to_index[lab]] for lab in cand_labels],
            dtype=np.float32
        )

        scores = scores * np.power(class_probs, beta)

        take = min(depth, len(cand))
        top_local = np.argpartition(-scores, take - 1)[:take]
        top_local = top_local[np.argsort(-scores[top_local])]
        nbrs = cand[top_local]

        rows.extend(build_run_rows_from_neighbors("soft_filtering", metric, i, nbrs, ids, y, depth))

    return pd.DataFrame(rows)

Cell 14 — Compute method weights from scratch (using average NDCG@10)


In [15]:

def compute_method_weights_for_embedding(X, y, ids):
    """
    Compute performance-aware method weights for Weighted RRF without Flat.

    Flat retrieval is intentionally excluded from the hybrid fusion and
    remains only an exact-search baseline in the paper.
    """
    ndcg_scores = {}

    metric = WEIGHT_TARGET_METRIC
    target_k = WEIGHT_TARGET_K

    # 1) soft filtering / classifier-guided
    run_soft = build_run_soft_filtering(
        X, y, ids, metric, FUSION_DEPTH,
        class_topk=CLASS_TOPK,
        beta=BETA,
        n_splits=N_SPLITS,
        random_state=RANDOM_STATE
    )
    perq_soft = evaluate_run(
        fuse_weighted_rrf(
            run_soft,
            weights={"soft_filtering": 1.0},
            rrf_k=RRF_K,
            method_name="soft_filtering"
        ),
        [target_k]
    )
    ndcg_scores["soft_filtering"] = perq_soft["ndcg@K"].mean()

    # 2) IVF
    run_ivf = build_run_ivf(
        X, y, ids, metric, FUSION_DEPTH,
        nlist=IVF_NLIST,
        nprobe=IVF_NPROBE
    )
    perq_ivf = evaluate_run(
        fuse_weighted_rrf(
            run_ivf,
            weights={"ivf": 1.0},
            rrf_k=RRF_K,
            method_name="ivf"
        ),
        [target_k]
    )
    ndcg_scores["ivf"] = perq_ivf["ndcg@K"].mean()

    # 3) IVFPQ
    run_ivfpq = build_run_ivfpq(
        X, y, ids, metric, FUSION_DEPTH,
        nlist=PQ_NLIST,
        nprobe=PQ_NPROBE,
        nbits=PQ_NBITS
    )
    perq_ivfpq = evaluate_run(
        fuse_weighted_rrf(
            run_ivfpq,
            weights={"ivfpq": 1.0},
            rrf_k=RRF_K,
            method_name="ivfpq"
        ),
        [target_k]
    )
    ndcg_scores["ivfpq"] = perq_ivfpq["ndcg@K"].mean()

    # 4) HNSW
    run_hnsw = build_run_hnsw(
        X, y, ids, metric, FUSION_DEPTH,
        M=HNSW_M,
        efc=HNSW_EF_CONSTRUCTION,
        efs=HNSW_EF_SEARCH
    )
    perq_hnsw = evaluate_run(
        fuse_weighted_rrf(
            run_hnsw,
            weights={"hnsw": 1.0},
            rrf_k=RRF_K,
            method_name="hnsw"
        ),
        [target_k]
    )
    ndcg_scores["hnsw"] = perq_hnsw["ndcg@K"].mean()

    # 5) filtered static
    if INCLUDE_FILTERED_STATIC:
        run_filtered = build_run_filtered_static(X, y, ids, metric, FUSION_DEPTH)
        perq_filtered = evaluate_run(
            fuse_weighted_rrf(
                run_filtered,
                weights={"filtered_static": 1.0},
                rrf_k=RRF_K,
                method_name="filtered_static"
            ),
            [target_k]
        )
        ndcg_scores["filtered_static"] = perq_filtered["ndcg@K"].mean()

    # Safety check: Flat must not appear in the weight table
    if "flat" in ndcg_scores:
        raise ValueError("Flat must not be included in Weighted RRF no-flat weights.")

    total = sum(ndcg_scores.values())
    if total <= 0:
        raise ValueError("Cannot compute method weights because total NDCG score is zero.")

    weights = {m: s / total for m, s in ndcg_scores.items()}

    return weights, ndcg_scores


Cell 15 — Weighted RRF fusion function


In [16]:

def fuse_weighted_rrf(run_df, weights, rrf_k=60, method_name=HYBRID_METHOD_NAME):
    """
    Weighted Reciprocal Rank Fusion.

    This version is configured for the no-Flat hybrid setting:
    Flat is excluded from the fusion and remains only an exact-search baseline.
    """
    df = run_df.copy()

    # Keep only fusion methods that are present in the weight dictionary
    df = df[df["method"].isin(weights.keys())].copy()

    # Safety check
    if "flat" in df["method"].unique():
        raise ValueError("Flat detected in Weighted RRF fusion. Remove it before fusion.")

    # Assign method weights
    df["method_weight"] = df["method"].map(weights)

    if df["method_weight"].isna().any():
        missing = sorted(df.loc[df["method_weight"].isna(), "method"].unique())
        raise ValueError(f"Missing weights for methods: {missing}")

    # Weighted reciprocal rank score
    df["wrrf_score"] = df["method_weight"] * (1.0 / (rrf_k + df["rank"].astype(float)))

    fused = (
        df.groupby(
            ["query_id", "query_label", "doc_id", "doc_label"],
            as_index=False
        )["wrrf_score"]
        .sum()
        .sort_values(["query_id", "wrrf_score"], ascending=[True, False])
    )

    fused["rank"] = (
        fused.groupby("query_id")["wrrf_score"]
        .rank(method="first", ascending=False)
        .astype(int)
    )

    fused["method"] = method_name

    return fused[["method", "query_id", "query_label", "doc_id", "doc_label", "rank", "wrrf_score"]]


Cell 16a — Define evaluation function


In [17]:
def evaluate_run(run_df, topk_list):

    rows = []

    for qid, g in run_df.groupby("query_id"):

        g = g.sort_values("rank")

        rels = (g["doc_label"].to_numpy() == g["query_label"].iloc[0]).astype(np.int32)
        total_rel = int(np.sum(rels))

        for K in topk_list:

            Ke = min(K, len(rels))

            rows.append({
                "query_id": qid,
                "query_label": int(g["query_label"].iloc[0]),
                "K": int(K),
                "precision@K": precision_at_k(rels, Ke),
                "recall@K": recall_at_k(rels, Ke, total_rel),
                "ndcg@K": ndcg_at_k(rels, Ke),
                "mrr@K": mrr_at_k(rels, Ke),
            })

    return pd.DataFrame(rows)

Cell 16 — Run Hybrid Method 2 (Performance-Weighted RRF without Flat) for all embeddings and all similarity metrics


In [ ]:

all_weighted_runs = []
all_weighted_perquery = []
all_weighted_summary = []
all_weight_tables = []

for emb_name, path in EMBEDDING_FILES.items():
    print(f"\n=== Hybrid Weighted RRF without Flat for embedding: {emb_name} ===")

    if not os.path.exists(path):
        print("File not found:", path)
        continue

    _, X, y, ids = load_embedding_xlsx(path)

    # --- compute weights from scratch for this embedding ---
    print("  -> computing method weights from NDCG@10 (cosine), excluding Flat")
    t0 = time.time()
    weights, ndcg_scores = compute_method_weights_for_embedding(X, y, ids)

    if "flat" in weights:
        raise ValueError("Flat appeared in weights. It must be excluded from Weighted RRF no-flat.")

    wt_df = pd.DataFrame({
        "embedding": emb_name,
        "method": list(weights.keys()),
        "weight": list(weights.values()),
        "ndcg@10": [ndcg_scores[m] for m in weights.keys()]
    })
    all_weight_tables.append(wt_df)

    print("     weights:", weights)
    print(f"     weight computation done in {round(time.time() - t0, 2)} sec")

    # --- run weighted fusion for all similarity metrics ---
    for metric in SIM_METRICS:
        print(f"  -> metric: {metric}")
        t0 = time.time()

        run_parts = []

        # 1) classifier-guided / soft filtering
        run_parts.append(
            build_run_soft_filtering(
                X, y, ids, metric, FUSION_DEPTH,
                class_topk=CLASS_TOPK,
                beta=BETA,
                n_splits=N_SPLITS,
                random_state=RANDOM_STATE
            )
        )

        # Flat is intentionally excluded from scalable hybrid fusion.

        # 2) IVF
        run_parts.append(
            build_run_ivf(
                X, y, ids, metric, FUSION_DEPTH,
                nlist=IVF_NLIST,
                nprobe=IVF_NPROBE
            )
        )

        # 3) IVFPQ
        run_parts.append(
            build_run_ivfpq(
                X, y, ids, metric, FUSION_DEPTH,
                nlist=PQ_NLIST,
                nprobe=PQ_NPROBE,
                nbits=PQ_NBITS
            )
        )

        # 4) HNSW
        run_parts.append(
            build_run_hnsw(
                X, y, ids, metric, FUSION_DEPTH,
                M=HNSW_M,
                efc=HNSW_EF_CONSTRUCTION,
                efs=HNSW_EF_SEARCH
            )
        )

        # 5) filtered static
        if INCLUDE_FILTERED_STATIC:
            run_parts.append(
                build_run_filtered_static(X, y, ids, metric, FUSION_DEPTH)
            )

        runs = pd.concat(run_parts, ignore_index=True)

        # Safety check: Flat must not be included
        if "flat" in runs["method"].unique():
            raise ValueError("Flat detected in run_parts. It must be excluded from Weighted RRF no-flat.")

        runs.insert(0, "embedding", emb_name)

        # weighted fusion
        fused = fuse_weighted_rrf(runs, weights, rrf_k=RRF_K, method_name=HYBRID_METHOD_NAME)
        fused.insert(0, "embedding", emb_name)
        fused.insert(2, "metric", metric)
        all_weighted_runs.append(fused)

        # evaluate
        perq = evaluate_run(fused, TOPK_LIST)
        perq.insert(0, "embedding", emb_name)
        perq.insert(1, "method", HYBRID_METHOD_NAME)
        perq.insert(2, "metric", metric)
        all_weighted_perquery.append(perq)

        # summary
        summary = (
            perq.groupby(["K"], as_index=False)
            .agg({
                "precision@K": "mean",
                "recall@K": "mean",
                "ndcg@K": "mean",
                "mrr@K": "mean"
            })
        )
        summary.insert(0, "embedding", emb_name)
        summary.insert(1, "method", HYBRID_METHOD_NAME)
        summary.insert(2, "metric", metric)
        all_weighted_summary.append(summary)

        print(f"     done in {round(time.time() - t0, 2)} sec")

df_weighted_run = pd.concat(all_weighted_runs, ignore_index=True)
df_weighted_perquery = pd.concat(all_weighted_perquery, ignore_index=True)
df_weighted_summary = pd.concat(all_weighted_summary, ignore_index=True)
df_weight_table = pd.concat(all_weight_tables, ignore_index=True)

print("\nShapes:")
print("HybridRun   :", df_weighted_run.shape)
print("PerQuery    :", df_weighted_perquery.shape)
print("Summary     :", df_weighted_summary.shape)
print("WeightTable :", df_weight_table.shape)

df_weighted_summary.head(20)



=== Hybrid Weighted RRF without Flat for embedding: bert_finetuned ===
  -> computing method weights from NDCG@10 (cosine), excluding Flat
     weights: {'soft_filtering': np.float64(0.19948917970956934), 'ivf': np.float64(0.19712157093175942), 'ivfpq': np.float64(0.19677395511764592), 'hnsw': np.float64(0.1968571496605759), 'filtered_static': np.float64(0.20975814458044936)}
     weight computation done in 20.42 sec
  -> metric: cosine
     done in 27.58 sec
  -> metric: dot
     done in 30.53 sec
  -> metric: neg_l2
     done in 35.33 sec

=== Hybrid Weighted RRF without Flat for embedding: gemini ===
  -> computing method weights from NDCG@10 (cosine), excluding Flat
     weights: {'soft_filtering': np.float64(0.1918899360408241), 'ivf': np.float64(0.19492938707656973), 'ivfpq': np.float64(0.194620298099907), 'hnsw': np.float64(0.19482683799222347), 'filtered_static': np.float64(0.22373354079047578)}
     weight computation done in 27.23 sec
  -> metric: cosine
     done in 16.43 s

,embedding,method,metric,K,precision@K,recall@K,ndcg@K,mrr@K
0,bert_finetuned,hybrid_weighted_rrf_no_flat,cosine,1,0.979337,0.015303,0.979337,0.979337
1,bert_finetuned,hybrid_weighted_rrf_no_flat,cosine,5,0.970039,0.075603,0.971924,0.981260
2,bert_finetuned,hybrid_weighted_rrf_no_flat,cosine,10,0.964744,0.150232,0.967604,0.981747
3,bert_finetuned,hybrid_weighted_rrf_no_flat,cosine,20,0.958760,0.297896,0.962368,0.981997
4,bert_finetuned,hybrid_weighted_rrf_no_flat,dot,1,0.978907,0.015280,0.978907,0.978907
5,bert_finetuned,hybrid_weighted_rrf_no_flat,dot,5,0.969350,0.075459,0.971197,0.980284
6,bert_finetuned,hybrid_weighted_rrf_no_flat,dot,10,0.963883,0.149882,0.966793,0.981105
7,bert_finetuned,hybrid_weighted_rrf_no_flat,dot,20,0.958825,0.297562,0.962189,0.981308
8,bert_finetuned,hybrid_weighted_rrf_no_flat,neg_l2,1,0.986225,0.015439,0.986225,0.986225
9,bert_finetuned,hybrid_weighted_rrf_no_flat,neg_l2,5,0.968833,0.075387,0.972692,0.987674


Cell 17 — Save results (Hybrid Weighted RRF without Flat)


In [19]:

# save summary + per-query + weights in Excel
with pd.ExcelWriter(OUT_SUMMARY_XLSX, engine="xlsxwriter") as writer:

    df_weighted_summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    df_weighted_perquery.to_excel(
        writer,
        sheet_name="PerQuery",
        index=False
    )

    df_weight_table.to_excel(
        writer,
        sheet_name="MethodWeights",
        index=False
    )

print("Saved:", OUT_SUMMARY_XLSX)


# save large ranked list as CSV
df_weighted_run.to_csv(
    OUT_RUN_CSV,
    index=False
)

print("Saved:", OUT_RUN_CSV)


Saved: hybrid_weighted_rrf_no_flat_summary.xlsx
Saved: hybrid_weighted_rrf_no_flat_run.csv
